# Exercises — Training at Scale

8 hands-on exercises covering every major concept from this chapter. Each exercise has:
1. **Problem statement** with exact numbers
2. **Scaffolded code** — fill in the blanks
3. **Full solution** with verification

Prerequisites: Run the setup cell below first.

In [ ]:
# === Setup ===
import numpy as np
np.random.seed(42)

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

def print_vector(name, v, decimals=6):
    formatted = ', '.join(f'{x:.{decimals}f}' for x in v)
    print(f"  {name} = [{formatted}]")

def print_table(headers, rows):
    widths = [max(len(str(h)), max(len(str(r[i])) for r in rows)) + 2 for i, h in enumerate(headers)]
    header_str = ''.join(f'{h:<{w}}' for h, w in zip(headers, widths))
    print(header_str)
    print('-' * sum(widths))
    for row in rows:
        print(''.join(f'{str(v):<{w}}' for v, w in zip(row, widths)))

print("Setup complete ✓")

## Exercise 1: Adam Update by Hand

**Given:**
- Gradient: $\mathbf{g} = [0.5, -1.2, 0.3]$
- Hyperparameters: $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\eta = 0.001$, $\epsilon = 10^{-8}$
- Time step: $t = 1$ (first step, so $m_0 = v_0 = \mathbf{0}$)

**Compute by hand:**
1. First moment $m_1$
2. Second moment $v_1$
3. Bias-corrected $\hat{m}_1$ and $\hat{v}_1$
4. Parameter update $\Delta\theta$

**Then:** Run for $t = 1$ through $t = 5$ and observe how bias correction changes.

In [ ]:
# Exercise 1: Adam Update — YOUR CODE

g = np.array([0.5, -1.2, 0.3])
beta1, beta2, eta, eps = 0.9, 0.999, 0.001, 1e-8

# Initialise moments
m = np.zeros(3)
v = np.zeros(3)

for t in range(1, 6):
    # TODO: Compute m_t = beta1 * m + (1 - beta1) * g
    # m = ???
    
    # TODO: Compute v_t = beta2 * v + (1 - beta2) * g^2
    # v = ???
    
    # TODO: Bias correction
    # m_hat = ???
    # v_hat = ???
    
    # TODO: Compute update
    # delta_theta = ???
    
    # print(f"t={t}: m_hat={m_hat}, delta_theta={delta_theta}")
    pass

print("Fill in the TODOs above and run!")

In [ ]:
# Exercise 1: SOLUTION

g = np.array([0.5, -1.2, 0.3])
beta1, beta2, eta, eps = 0.9, 0.999, 0.001, 1e-8

m = np.zeros(3)
v = np.zeros(3)

print("=== Adam Step-by-Step Solution ===\n")
print(f"{'t':<4} {'m₁':<12} {'v₁':<14} {'m̂₁':<12} {'v̂₁':<14} {'Δθ₁':<14} {'1-β₁ᵗ':<10}")
print("-" * 80)

for t in range(1, 6):
    # First moment
    m = beta1 * m + (1 - beta1) * g
    # Second moment
    v = beta2 * v + (1 - beta2) * g**2
    # Bias correction
    m_hat = m / (1 - beta1**t)
    v_hat = v / (1 - beta2**t)
    # Update
    delta_theta = eta * m_hat / (np.sqrt(v_hat) + eps)
    
    print(f"{t:<4} {m[0]:<12.6f} {v[0]:<14.9f} {m_hat[0]:<12.6f} {v_hat[0]:<14.9f} "
          f"{delta_theta[0]:<14.9f} {1-beta1**t:<10.6f}")

print(f"\nKey observations:")
print(f"  • At t=1: m̂ = g (bias correction perfectly undoes the (1-β₁) scaling)")
print(f"  • At t=5: m̂ ≈ g (bias correction factor 1-β₁⁵ = {1-beta1**5:.4f} ≈ 0.41)")
print(f"  • Δθ ≈ η for all t (Adam normalises updates to ~unit scale)")

## Exercise 2: Gradient Clipping

**Given:** $\mathbf{g} = [3.0, 4.0, 0.0]$, threshold $\tau = 1.0$

**Compute:**
1. $\|\mathbf{g}\|_2$
2. The clipped gradient $\mathbf{g}_{\text{clip}}$
3. Verify $\|\mathbf{g}_{\text{clip}}\|_2 = \tau$
4. Verify the direction is preserved: $\cos(\mathbf{g}, \mathbf{g}_{\text{clip}}) = 1$

**Then:** Compare clip-by-norm vs clip-by-value on $\mathbf{g} = [3.0, 4.0, 0.0]$.

In [ ]:
# Exercise 2: Gradient Clipping — YOUR CODE

g = np.array([3.0, 4.0, 0.0])
tau = 1.0

# TODO: Compute gradient norm
# norm = ???

# TODO: Clip by norm (preserves direction)
# if norm > tau:
#     g_clipped = ???
# else:
#     g_clipped = g.copy()

# TODO: Verify
# print(f"‖g‖ = {norm}")
# print(f"g_clipped = {g_clipped}")
# print(f"‖g_clipped‖ = {np.linalg.norm(g_clipped)}")

# TODO: Compare with clip-by-value
# g_value_clipped = np.clip(g, -tau, tau)
# cos_norm = ???
# cos_value = ???

print("Fill in the TODOs above and run!")

In [ ]:
# Exercise 2: SOLUTION

g = np.array([3.0, 4.0, 0.0])
tau = 1.0

# 1. Gradient norm
norm = np.linalg.norm(g)
print(f"‖g‖₂ = √(3² + 4² + 0²) = √(9 + 16) = √25 = {norm:.1f}")
print(f"Since {norm} > τ = {tau}, clipping is triggered.\n")

# 2. Clip by norm
g_clipped = tau * g / norm
print(f"g_clip = τ × g/‖g‖ = {tau} × {g} / {norm}")
print_vector("g_clip", g_clipped)

# 3. Verify norm
print(f"\n‖g_clip‖ = {np.linalg.norm(g_clipped):.4f} = τ = {tau} ✓")

# 4. Direction preservation
cos_sim = np.dot(g, g_clipped) / (np.linalg.norm(g) * np.linalg.norm(g_clipped))
print(f"cos(g, g_clip) = {cos_sim:.6f} = 1.0 ✓ (direction preserved)")

# 5. Compare with clip-by-value
g_value = np.clip(g, -tau, tau)
print(f"\n--- Clip-by-value comparison ---")
print_vector("g_value", g_value)
cos_value = np.dot(g, g_value) / (np.linalg.norm(g) * np.linalg.norm(g_value))
print(f"cos(g, g_value) = {cos_value:.6f} ≠ 1.0 → direction DISTORTED")
print(f"‖g_value‖ = {np.linalg.norm(g_value):.4f} ≠ τ → magnitude wrong too")
print(f"\n→ Clip-by-norm is strictly better: preserves direction AND controls magnitude.")

## Exercise 3: ZeRO Memory Calculation

**Given:** A 7B parameter model, 8 GPUs, mixed-precision training (fp16 params + fp32 optimizer).

**Calculate per-GPU memory for each ZeRO stage:**
- No ZeRO: params (2N) + gradients (2N) + optimizer (12N)
- ZeRO-1: Shard optimizer states
- ZeRO-2: Shard optimizer + gradients
- ZeRO-3: Shard everything

**Then:** Determine the minimum number of A100-80GB GPUs needed for a 70B model at each ZeRO stage.

In [ ]:
# Exercise 3: ZeRO Memory — YOUR CODE

N = 7e9   # 7B parameters
K = 8     # 8 GPUs
GB = 1024**3

# TODO: Calculate per-GPU memory for each ZeRO stage
# No ZeRO: params + grads + optimizer = 2N + 2N + 12N = 16N bytes
# no_zero = ???

# ZeRO-1: Shard optimizer only → 2N + 2N + 12N/K
# zero1 = ???

# ZeRO-2: Shard optimizer + gradients → 2N + 2N/K + 12N/K  
# zero2 = ???

# ZeRO-3: Shard everything → 16N/K
# zero3 = ???

# print(f"No ZeRO: {no_zero / GB:.1f} GB")
# print(f"ZeRO-1:  {zero1 / GB:.1f} GB")
# print(f"ZeRO-2:  {zero2 / GB:.1f} GB")
# print(f"ZeRO-3:  {zero3 / GB:.1f} GB")

print("Fill in the TODOs above and run!")

In [ ]:
# Exercise 3: SOLUTION

N = 7e9
K = 8
GB = 1024**3

print("=== ZeRO Memory: 7B Model, 8 GPUs ===\n")
print("Memory formulas (bytes):")
print("  Params:    2N (fp16)")
print("  Gradients: 2N (fp16)")
print("  Optimizer: 4N (fp32 copy) + 4N (m) + 4N (v) = 12N\n")

stages = {
    'No ZeRO': 2*N + 2*N + 12*N,
    'ZeRO-1':  2*N + 2*N + 12*N/K,
    'ZeRO-2':  2*N + 2*N/K + 12*N/K,
    'ZeRO-3':  2*N/K + 2*N/K + 12*N/K,
}

print(f"{'Stage':<12} {'Formula':<30} {'GB per GPU':<14} {'Reduction'}")
print("-" * 66)
base = stages['No ZeRO']
for name, mem in stages.items():
    formula = {'No ZeRO': '2N + 2N + 12N = 16N',
               'ZeRO-1': '2N + 2N + 12N/K',
               'ZeRO-2': '2N + 2N/K + 12N/K',
               'ZeRO-3': '16N/K'}[name]
    print(f"{name:<12} {formula:<30} {mem/GB:<14.1f} {base/mem:.1f}×")

# 70B model: minimum GPUs
print("\n\n=== Minimum A100-80GB GPUs for 70B Model ===\n")
N70 = 70e9
usable = 70 * GB  # ~70GB usable out of 80GB

for name in ['No ZeRO', 'ZeRO-1', 'ZeRO-2', 'ZeRO-3']:
    if name == 'No ZeRO':
        per_gpu = lambda K: (2 + 2 + 12) * N70  # doesn't shard
    elif name == 'ZeRO-1':
        per_gpu = lambda K: (2 + 2) * N70 + 12 * N70 / K
    elif name == 'ZeRO-2':
        per_gpu = lambda K: 2 * N70 + (2 + 12) * N70 / K
    else:
        per_gpu = lambda K: 16 * N70 / K
    
    # Find minimum K
    for K in range(1, 1025):
        if per_gpu(K) <= usable:
            print(f"{name}: {K} GPUs (per-GPU: {per_gpu(K)/GB:.1f} GB)")
            break
    else:
        print(f"{name}: >1024 GPUs needed")

## Exercise 4: Pipeline Parallelism Bubble

**Given:** $p = 4$ pipeline stages.

**Calculate:**
1. Bubble fraction for $m = 8$ microbatches: $f = \frac{p-1}{p-1+m}$
2. Bubble fraction for $m = 16$ microbatches
3. Effective throughput for both (tokens/sec if no-bubble throughput = 1000 tokens/sec)
4. How many microbatches needed for $< 5\%$ bubble overhead?

In [ ]:
# Exercise 4: Pipeline Bubble — YOUR CODE

p = 4  # pipeline stages

# TODO: Bubble fraction for m=8
# f_8 = (p - 1) / (p - 1 + 8)
# print(f"m=8:  bubble = {f_8:.2%}")

# TODO: Bubble fraction for m=16
# f_16 = ???

# TODO: Effective throughput (base = 1000 tokens/sec)
# throughput_8 = 1000 * (1 - f_8)
# throughput_16 = ???

# TODO: Find minimum m for <5% bubble
# for m in range(1, 200):
#     if (p-1)/(p-1+m) < 0.05:
#         print(f"Need m >= {m}")
#         break

print("Fill in the TODOs above and run!")

In [ ]:
# Exercise 4: SOLUTION

p = 4
base_throughput = 1000  # tokens/sec

print("=== Pipeline Bubble Analysis (p=4 stages) ===\n")
print(f"Formula: f_bubble = (p-1) / (p-1+m) = {p-1} / ({p-1}+m)\n")

# Part 1-3
for m in [8, 16]:
    f = (p - 1) / (p - 1 + m)
    throughput = base_throughput * (1 - f)
    print(f"m = {m}:")
    print(f"  f_bubble = {p-1}/({p-1}+{m}) = {p-1}/{p-1+m} = {f:.4f} = {f:.1%}")
    print(f"  Efficiency = 1 - {f:.4f} = {1-f:.1%}")
    print(f"  Throughput = {base_throughput} × {1-f:.4f} = {throughput:.0f} tokens/sec")
    print()

# Part 4: Find minimum m for <5% bubble
print("--- Minimum m for <5% bubble ---")
print(f"Solve: {p-1}/({p-1}+m) < 0.05")
print(f"  {p-1} < 0.05 × ({p-1}+m)")
print(f"  {p-1} < {0.05*(p-1)} + 0.05m")
print(f"  {p-1 - 0.05*(p-1)} < 0.05m")
print(f"  m > {(p-1 - 0.05*(p-1))/0.05:.0f}")
m_min = int(np.ceil((p-1) / 0.05 - (p-1)))
print(f"  → m ≥ {m_min} microbatches")
print(f"\nVerification: f({m_min}) = {(p-1)/(p-1+m_min):.4f} = {(p-1)/(p-1+m_min):.2%}")

## Exercise 5: MFU Calculation

**Given:**
- 1024 A100 GPUs (peak: 312 TFLOP/s each in bf16)
- 7B parameter model
- 2M tokens per training step
- 1.2 seconds per step

**Calculate:**
1. FLOPs per step ($6N \times T$)
2. Achieved FLOP/s
3. Peak cluster FLOP/s
4. MFU percentage

**Then:** What MFU would you get with the same setup on H100s (989 TFLOP/s)?

In [ ]:
# Exercise 5: MFU — YOUR CODE

N = 7e9          # 7B parameters
T = 2e6          # 2M tokens per step
t_step = 1.2     # seconds per step
G = 1024         # number of GPUs
F_a100 = 312e12  # A100 peak FLOP/s
F_h100 = 989e12  # H100 peak FLOP/s

# TODO: FLOPs per step = 6 × N × T
# flops_per_step = ???

# TODO: Achieved FLOP/s = flops_per_step / t_step
# achieved = ???

# TODO: Peak FLOP/s = G × F_peak
# peak_a100 = ???

# TODO: MFU = achieved / peak
# mfu_a100 = ???

# TODO: Same on H100
# peak_h100 = ???
# mfu_h100 = ???

print("Fill in the TODOs above and run!")

In [ ]:
# Exercise 5: SOLUTION

N = 7e9; T = 2e6; t_step = 1.2; G = 1024
F_a100 = 312e12; F_h100 = 989e12

print("=== MFU Calculation ===\n")

# Step 1: FLOPs per step
flops_per_step = 6 * N * T
print(f"1. FLOPs/step = 6 × N × T")
print(f"   = 6 × {N:.0e} × {T:.0e}")
print(f"   = {flops_per_step:.2e}")

# Step 2: Achieved FLOP/s
achieved = flops_per_step / t_step
print(f"\n2. Achieved = {flops_per_step:.2e} / {t_step}")
print(f"   = {achieved:.2e} FLOP/s")

# Step 3: Peak A100
peak_a100 = G * F_a100
print(f"\n3. Peak (A100) = {G} × {F_a100:.0e}")
print(f"   = {peak_a100:.2e} FLOP/s")

# Step 4: MFU
mfu_a100 = achieved / peak_a100
print(f"\n4. MFU (A100) = {achieved:.2e} / {peak_a100:.2e}")
print(f"   = {mfu_a100:.1%}")

# Bonus: H100
peak_h100 = G * F_h100
mfu_h100 = achieved / peak_h100
print(f"\n--- Same setup on H100 ---")
print(f"Peak (H100) = {peak_h100:.2e}")
print(f"MFU (H100) = {mfu_h100:.1%}")
print(f"\n→ Same achieved FLOP/s but higher peak → lower MFU")
print(f"   This is why MFU should be compared within the same hardware class")

## Exercise 6: LoRA Parameter Count

**Given:** LLaMA-7B architecture:
- Hidden dimension $d = 4096$
- LoRA rank $r = 16$
- Target modules: Q and V projections
- Layers: 32

**Calculate:**
1. LoRA parameters per adapted weight: $2 \times d \times r$
2. Total LoRA parameters (Q + V across all layers)
3. Full model parameter count (approximately)
4. Percentage of trainable parameters
5. Repeat for $r = 4, 8, 32, 64$

In [ ]:
# Exercise 6: LoRA Parameters — YOUR CODE

d = 4096      # hidden dimension
n_layers = 32
full_params = 7e9  # approximate

# TODO: For r=16, Q+V target
# r = 16
# params_per_module = 2 * d * r   # B(d×r) + A(r×d)
# n_modules = 2   # Q and V
# total_lora = params_per_module * n_modules * n_layers
# pct = total_lora / full_params * 100

# TODO: Build table for r = 4, 8, 16, 32, 64

print("Fill in the TODOs above and run!")

In [ ]:
# Exercise 6: SOLUTION

d = 4096
n_layers = 32
full_params = 7e9

print("=== LoRA Parameter Count: LLaMA-7B ===\n")

# Detailed calculation for r=16
r = 16
params_per_module = 2 * d * r  # B: d×r, A: r×d
n_modules = 2  # Q and V

print(f"Per adapted weight: 2 × d × r = 2 × {d} × {r} = {params_per_module:,}")
print(f"Per layer: {params_per_module:,} × {n_modules} modules = {params_per_module * n_modules:,}")
print(f"Total: {params_per_module * n_modules:,} × {n_layers} layers = {params_per_module * n_modules * n_layers:,}")
print(f"Percentage: {params_per_module * n_modules * n_layers / full_params * 100:.3f}%\n")

# Full table
print(f"{'Rank r':<10} {'LoRA Params':<18} {'% of 7B':<12} {'Reduction':<12} {'Memory (fp16)'}")
print("-" * 64)
for r in [4, 8, 16, 32, 64]:
    total_lora = 2 * d * r * 2 * n_layers  # 2 modules (Q, V)
    pct = total_lora / full_params * 100
    reduction = full_params / total_lora
    mem_mb = total_lora * 2 / (1024**2)  # fp16 = 2 bytes
    print(f"{r:<10} {total_lora:>14,}   {pct:<12.4f} {reduction:>8,.0f}×{'':<3} {mem_mb:.1f} MB")

# Extended: different target modules
print(f"\n--- r=16, varying target modules ---\n")
print(f"{'Target':<22} {'Modules':<10} {'Params':<14} {'% of 7B'}")
print("-" * 56)
for name, n_mod in [("Q, V", 2), ("Q, K, V, O", 4), ("Attn + FFN", 6)]:
    total = 2 * d * 16 * n_mod * n_layers
    print(f"{name:<22} {n_mod:<10} {total:>10,}   {total/full_params*100:.4f}%")

## Exercise 7: Critical Batch Size

**The critical batch size** balances gradient noise vs computation:
$$B_{\text{crit}} = \frac{B_{\text{noise}}}{B_{\text{signal}}}$$

**Given:** $B_{\text{noise}} = 10^6$ tokens, $B_{\text{signal}} = 2 \times 10^4$ tokens.

**Calculate:**
1. Critical batch size
2. Time-optimal batch size (about $B_{\text{crit}}$)
3. Compute-optimal batch size (about $B_{\text{crit}} / 2$)
4. If $B < B_{\text{crit}}$: more steps needed (time-limited)
5. If $B > B_{\text{crit}}$: diminishing returns (compute-limited)

**Then:** Plot the trade-off between total compute and wall-clock time as batch size varies.

In [ ]:
# Exercise 7: Critical Batch Size — YOUR CODE

B_noise = 1e6
B_signal = 2e4

# TODO: Critical batch size
# B_crit = B_noise / B_signal

# TODO: Time-optimal and compute-optimal
# B_time_opt = B_crit
# B_compute_opt = B_crit / 2

# TODO: For various batch sizes, compute:
#   steps needed ∝ 1 + B_crit/B
#   total compute ∝ B × steps = B × (1 + B_crit/B) = B + B_crit

print("Fill in the TODOs above and run!")

In [ ]:
# Exercise 7: SOLUTION

B_noise = 1e6
B_signal = 2e4

B_crit = B_noise / B_signal
print("=== Critical Batch Size ===\n")
print(f"B_crit = B_noise / B_signal = {B_noise:.0e} / {B_signal:.0e} = {B_crit:.0f} tokens")
print(f"\nTime-optimal batch size:    ~B_crit = {B_crit:.0f}")
print(f"Compute-optimal batch size: ~B_crit/2 = {B_crit/2:.0f}")

# Trade-off analysis
print(f"\n{'Batch Size':<14} {'B/B_crit':<12} {'Steps (×S_min)':<18} {'Compute (×C_min)':<18} {'Regime'}")
print("-" * 74)

# Simplified model: steps ∝ (1 + B_crit/B), compute ∝ B × (1 + B_crit/B)
S_min = 1.0  # normalised
for B in [B_crit/10, B_crit/4, B_crit/2, B_crit, 2*B_crit, 4*B_crit, 10*B_crit]:
    ratio = B / B_crit
    steps = 1 + B_crit / B  # relative to minimum
    compute = B * steps      # relative
    compute_norm = compute / (2 * B_crit)  # normalise
    steps_norm = steps / 2   # normalise (at B=B_crit, steps=2)
    regime = "time-limited" if B < B_crit else ("sweet spot" if B == B_crit else "compute-limited")
    print(f"{B:<14.0f} {ratio:<12.2f} {steps_norm:<18.2f} {compute_norm:<18.2f} {regime}")

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(8, 5))
    B_range = np.logspace(np.log10(B_crit/20), np.log10(B_crit*20), 200)
    steps_rel = 1 + B_crit / B_range
    compute_rel = B_range * steps_rel / (2 * B_crit)
    steps_norm = steps_rel / 2

    ax.plot(B_range / B_crit, steps_norm, 'b-', linewidth=2, label='Steps (time)')
    ax.plot(B_range / B_crit, compute_rel, 'r-', linewidth=2, label='Compute')
    ax.axvline(1.0, color='green', linestyle='--', alpha=0.5, label='B = B_crit')
    ax.set_xlabel('B / B_crit'); ax.set_ylabel('Relative Cost')
    ax.set_title('Critical Batch Size Trade-Off')
    ax.set_xscale('log'); ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('critical_batch.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: critical_batch.png")

## Exercise 8: Cosine Schedule Computation

**Given:**
- $\eta_{\max} = 3 \times 10^{-4}$, $\eta_{\min} = 3 \times 10^{-5}$
- Warmup $W = 2000$ steps, Total $T = 100{,}000$ steps

**Compute $\eta_t$ at:**
1. $t = 1000$ (warmup phase)
2. $t = 5000$ (early cosine)
3. $t = 50{,}000$ (mid cosine)
4. $t = 99{,}000$ (near end)

**Formula:**
$$\eta_t = \begin{cases} \eta_{\max} \cdot t/W & t < W \\ \eta_{\min} + \frac{1}{2}(\eta_{\max} - \eta_{\min})(1 + \cos(\pi \cdot \frac{t-W}{T-W})) & t \ge W \end{cases}$$

In [ ]:
# Exercise 8: Cosine Schedule — YOUR CODE

eta_max = 3e-4
eta_min = 3e-5
W = 2000
T = 100000

test_points = [1000, 5000, 50000, 99000]

# TODO: Implement cosine schedule
# def cosine_lr(t, eta_max, eta_min, W, T):
#     if t < W:
#         return ???  # linear warmup
#     else:
#         progress = ???
#         return ???  # cosine decay

# for t in test_points:
#     lr = cosine_lr(t, eta_max, eta_min, W, T)
#     print(f"t={t}: η = {lr:.6e}")

print("Fill in the TODOs above and run!")

In [ ]:
# Exercise 8: SOLUTION

eta_max = 3e-4
eta_min = 3e-5
W = 2000
T = 100000

def cosine_lr(t, eta_max, eta_min, W, T):
    if t < W:
        return eta_max * t / W
    progress = (t - W) / (T - W)
    return eta_min + 0.5 * (eta_max - eta_min) * (1 + np.cos(np.pi * progress))

print("=== Cosine LR Schedule ===\n")
print(f"η_max = {eta_max:.0e}, η_min = {eta_min:.0e}, W = {W}, T = {T:,}\n")

test_points = [1000, 5000, 50000, 99000]

for t in test_points:
    lr = cosine_lr(t, eta_max, eta_min, W, T)
    phase = "WARMUP" if t < W else "COSINE"
    if t < W:
        calc = f"η_max × t/W = {eta_max:.0e} × {t}/{W} = {eta_max * t / W:.6e}"
    else:
        progress = (t - W) / (T - W)
        cos_val = np.cos(np.pi * progress)
        calc = f"progress = {progress:.4f}, cos(π×{progress:.4f}) = {cos_val:.4f}"
    print(f"t = {t:>6,}  [{phase}]")
    print(f"  {calc}")
    print(f"  η_{t} = {lr:.6e} ({lr/eta_max:.1%} of η_max)\n")

# Plot full schedule
if HAS_MPL:
    steps = np.arange(T)
    lrs = [cosine_lr(t, eta_max, eta_min, W, T) for t in steps]
    
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(steps[::50], lrs[::50], 'b-', linewidth=2)
    for t in test_points:
        lr = cosine_lr(t, eta_max, eta_min, W, T)
        ax.plot(t, lr, 'ro', markersize=8)
        ax.annotate(f't={t:,}\nη={lr:.1e}', (t, lr), textcoords='offset points',
                   xytext=(10, 10), fontsize=8, color='red')
    ax.axvline(W, color='gray', linestyle='--', alpha=0.5, label=f'Warmup end (t={W})')
    ax.set_xlabel('Step'); ax.set_ylabel('Learning Rate')
    ax.set_title('Cosine LR Schedule with Warmup')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    ax.ticklabel_format(style='sci', axis='y', scilimits=(-4,-4))
    plt.tight_layout()
    plt.savefig('cosine_schedule_exercise.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: cosine_schedule_exercise.png")

print("\n=== Key Takeaways ===")
print(f"• Warmup: LR increases linearly from 0 to η_max over {W} steps")
print(f"• At t={W}: LR = η_max = {eta_max:.0e}")
print(f"• Mid-training (t=50K): LR ≈ midpoint = {(eta_max+eta_min)/2:.1e}")
print(f"• Near end (t=99K): LR ≈ η_min = {eta_min:.0e}")
print(f"• Cosine gives slow start, fast middle, slow end — smooth everywhere")